# Synthesise training data

This notebook is the synthesiser. It turns clean G-NAF records into mutated addresses for training the retriever and the re-ranker. It makes positives (a mutated address and the G-NAF record it should match) and negatives (a plausible address that matches no record). Every mistake rate is fitted on the test set's 789 matchable and 161 unmatchable rows. The method follows Yin, Li and Goldberg's component error injector and Kuznetsov and Urdiales' character engine. `err_` operations change a fact of the address (a wrong suburb). `fmt_` operations write the same fact another way (AVE for AVENUE).

- Reads: `output/misspellings.tsv`, `output/char_edit_ops.parquet` and `output/substitutions.tsv` from the harvest, and the `gnaf_key`, `gnaf_streets` and `gnaf_range` tables of `gnaf.duckdb`. Cell 11 also reads `output_stage1/stage1_recall_per_query.tsv`.
- Writes: `output_synthesis/synthetic_addresses.parquet` (100,000 rows) and `output_synthesis/sample_500.tsv`.
- Run after: 1_harvest_test_set.ipynb. Cell 11 runs after 3_train_retriever.ipynb.

Cell 5 reads every `gnaf_key` row into memory and takes about a minute. Cell 9 overwrites the training set.

In [ ]:
# Cell 1: configuration. Paths, column names, sizes, the seed and the
# generator's settings.

from pathlib import Path

DB_PATH   = Path("gnaf.duckdb")
EXPORT_IN = Path("output")
OUT       = Path("output_synthesis"); OUT.mkdir(exist_ok=True)

MATRIX_TSV  = EXPORT_IN / "mutation_matrix.tsv"   # not read by any cell
CHAR_SUBS   = EXPORT_IN / "substitutions.tsv"
CHAR_OPS_PQ = EXPORT_IN / "char_edit_ops.parquet"

COLMAP = dict(
    key_table="gnaf_key", streets_table="gnaf_streets",
    number="k", street="street_n", s_type="type_n", suffix="suffix_n",
    locality="locality_n", state="state_n", postcode="postcode",
    pid="pid",                  # gnaf_key primary key; joins gnaf_range
    range_table="gnaf_range",   # "" disables ranges
    alias_table="",             # locality alias table, if one exists; "" disables
)

SEED     = 4640
TARGET_N = 100_000   # mutated positives and negatives
N_CLEAN  = 0         # extra format-only positives
NEG_FRAC = 0.15      # share of TARGET_N that are negatives
BATCH    = 5_000     # records per DuckDB sample

P_SUB_EMPIRICAL = 0.80           # share of substitutions from the empirical table
ALLOW_FIRST_CHAR = False         # whether a typo may change the first letter
TRANSPOSE_PAIRS = 4              # swaps taken from harvest substitutions (cell 4)
P_ABBREV_GIVEN_ELIGIBLE = 0.90   # cap on the fitted abbreviation rates (cell 6b)
P_SUBLOCAL_DROP         = 0.55   # split of err_loc_sublocal between dropping a
                                 # geographic tail and appending a colloquial one

import numpy as np, pandas as pd, re, duckdb
rng = np.random.default_rng(SEED)
print("config ok")

config ok


In [ ]:
# Cell 2: open gnaf.duckdb read-only and print every table's columns.
con = duckdb.connect(str(DB_PATH), read_only=True)
for t in con.execute("SHOW TABLES").df().name:
    print(f"-- {t}")
    print(con.execute(f"DESCRIBE {t}").df().to_string(index=False), "\n")

-- core
       column_name column_type null  key default extra
ADDRESS_DETAIL_PID     VARCHAR  YES None    None  None
      DATE_CREATED     VARCHAR  YES None    None  None
     ADDRESS_LABEL     VARCHAR  YES None    None  None
 ADDRESS_SITE_NAME     VARCHAR  YES None    None  None
     BUILDING_NAME     VARCHAR  YES None    None  None
         FLAT_TYPE     VARCHAR  YES None    None  None
       FLAT_NUMBER     VARCHAR  YES None    None  None
        LEVEL_TYPE     VARCHAR  YES None    None  None
      LEVEL_NUMBER     VARCHAR  YES None    None  None
      NUMBER_FIRST     VARCHAR  YES None    None  None
       NUMBER_LAST     VARCHAR  YES None    None  None
        LOT_NUMBER     VARCHAR  YES None    None  None
       STREET_NAME     VARCHAR  YES None    None  None
       STREET_TYPE     VARCHAR  YES None    None  None
     STREET_SUFFIX     VARCHAR  YES None    None  None
     LOCALITY_NAME     VARCHAR  YES None    None  None
             STATE     VARCHAR  YES None    None  None
  

In [ ]:
# Cell 3: fit every rate the generator needs from the test set.
# Error flags come from the `reason` column, one reason string per error,
# separated by semicolons. Format flags are detected in the written address
# against the true G-NAF components. Each format rate is fitted given that the
# record is eligible for it, which is how render() applies it.

import re, numpy as np, pandas as pd
from pathlib import Path

MISS_TSV = next((p for p in [EXPORT_IN / "misspellings.tsv",
                             Path("output/misspellings.tsv"),
                             Path("misspellings.tsv")] if p.exists()), None)
assert MISS_TSV is not None, "misspellings.tsv not found"
miss = pd.read_csv(MISS_TSV, sep="\t", dtype=str).fillna("")
miss["rid"] = miss.index
print(f"read {len(miss):,} rows from {MISS_TSV}")

# 1. Reason strings to error flags.
REASON_TO_FLAG = {
    "Cannot be matched":                            "no_record",
    "Street number does not exist on this street":  "err_number",
    "Unit, level or lot detail differs":            "err_occupancy",
    "Postcode does not match":                      "err_postcode",
    "State does not match":                         "err_state",
    "Wrong suburb (a different real locality)":     "err_loc_wrong",
    "Suburb missing or gaining a direction word":   "err_loc_qualifier",
    "Suburb misspelt (1-2 characters)":             "err_loc_typo",
    "Suburb misspelt (3+ characters)":              "err_loc_heavy",
    "Suburb abbreviated":                           "err_loc_abbrev",
    "Suburb word spacing wrong":                    "err_loc_spacing",
    "Suburb is an unofficial or sub-locality name": "err_loc_sublocal",
    "Street type wrong":                            "err_st_type",
    "Street direction suffix wrong":                "err_st_suffix",
    "Street name misspelt (1 character)":           "err_st_edit1",
    "Street name misspelt (2 characters)":          "err_st_edit2",
    "Street name misspelt (3+ characters)":         "err_st_heavy",
    "Street name substantially different":          "err_st_different",
    "Street name plural or possessive S wrong":     "err_st_possessive",
    "Street name spacing or apostrophe wrong":      "err_st_spacing",
    "Street name missing or gaining a word":        "err_st_worddrop",
    "Street name abbreviated":                      "err_st_abbrev",
}
_KEYS = sorted(REASON_TO_FLAG, key=len, reverse=True)   # longest first

def _reason_flags(reason):
    got, unknown = set(), []
    for frag in str(reason).split(";"):
        frag = frag.strip()
        if not frag: continue
        for k in _KEYS:
            if frag.startswith(k):
                got.add(REASON_TO_FLAG[k]); break
        else:
            unknown.append(frag)
    return got, unknown

_parsed  = miss.reason.map(_reason_flags)
_unknown = sorted({u for _, us in _parsed for u in us})
assert not _unknown, ("reason strings this cell cannot map (add them to "
                      f"REASON_TO_FLAG before trusting any rate): {_unknown}")

ALL_FLAGS = list(dict.fromkeys(REASON_TO_FLAG.values()))
F = pd.DataFrame({c: _parsed.map(lambda s, c=c: c in s[0]) for c in ALL_FLAGS})
ERR_COLS = [c for c in ALL_FLAGS
            if c not in ("no_record", "err_number", "err_occupancy")]

is_pos = ~(F.no_record | F.err_number)
pos, neg = miss[is_pos].reset_index(drop=True), miss[~is_pos].reset_index(drop=True)
Fp = F[is_pos].reset_index(drop=True)
print(f"{len(pos)} matchable, {len(neg)} unmatchable "
      f"({int(F.no_record.sum())} no_record, {int(F.err_number.sum())} no_number)")

# 2. Draw weights (what the generator samples) and flag targets (what cell 10
# checks). A wrong suburb that also changes the postcode is one draw, two flags.
flag_counts = Fp[ERR_COLS].sum(); flag_counts = flag_counts[flag_counts > 0]
ERR_TARGET  = (flag_counts / flag_counts.sum()).to_dict()

drag_rows = Fp.err_loc_wrong & Fp.err_postcode
P_DRAG_PC = float(drag_rows.sum() / max(int(Fp.err_loc_wrong.sum()), 1))

draw_counts = flag_counts.copy()
draw_counts["err_postcode"] = int((Fp.err_postcode & ~Fp.err_loc_wrong).sum())
ERR_W = (draw_counts / draw_counts.sum()).to_dict()

n_flags = Fp[ERR_COLS].sum(axis=1)
n_draws = n_flags - drag_rows.astype(int)
hf = n_flags[n_flags > 0].value_counts().sort_index(); FLAG_TARGET_P = (hf/hf.sum()).to_dict()
hd = n_draws[n_draws > 0].value_counts().sort_index(); NDRAW_P      = (hd/hd.sum()).to_dict()

# 3. True G-NAF components for every matchable row, joined from gnaf_key
# rather than parsed from the label. Building names ("WESTFIELD SHOPPING
# CENTRE 101 GRAFTON ST") make parsing unreliable. Whether the record is a
# range comes from gnaf_range.
def _norm(s):
    s = str(s).upper().replace("\u2013", "-").replace("\u2014", "-").replace(",", " ")
    return re.sub(r"\s+", " ", s).strip()

pos["m"] = pos.matched.map(_norm)
con.register("miss_pos_df", pos[["rid", "m"]])
truth = con.sql(r"""
    SELECT m.rid, k.k, coalesce(g.number_last,'') AS number_last,
           k.street_n, k.type_n, k.locality_n, k.state_n, k.postcode
    FROM miss_pos_df m
    JOIN gnaf_key k
      ON regexp_replace(trim(replace(upper(k.ADDRESS_LABEL), ',', ' ')),
                        '\s+', ' ', 'g') = m.m
    LEFT JOIN gnaf_range g ON g.pid = k.pid AND g.number_first = k.k
    QUALIFY row_number() OVER (PARTITION BY m.rid
        ORDER BY CASE WHEN k.k LIKE 'LOT %' THEN 1 ELSE 0 END, k.pid) = 1
""").df()
P = pos.merge(truth, on="rid", how="inner")
print(f"G-NAF components resolved for {len(P):,} of {len(pos):,} matchable rows "
      f"({len(P)/max(len(pos),1):.1%}); fmt_ rates are fitted on these")
assert len(P) / max(len(pos), 1) > 0.95, "truth join is failing; fmt_ rates unreliable"

Fpi = Fp.set_index(pos.rid).loc[P.rid].reset_index(drop=True)   # flags aligned to P

# 4. fmt_ detectors. CONSUMES words take the number after them as a unit
# number. NOISE words are skipped. "Ward 6 Bellevarde Parade Mona Vale" matches
# G-NAF's "6 BELLEVARDE PDE", so WARD is noise. SHOP is right for
# "Shop 5/138 Pacific Hwy" and wrong for "Shop 302 Murray St", which then
# counts as unreadable below.
CONSUMES = r"(?:UNIT|APT|APARTMENT|FLAT|LEVEL|LVL|SHOP|SUITE|STE|ROOM|OFFICE|SITE)"
NOISE    = r"(?:HOUSE|BUILDING|TOWER|BLOCK|REAR|WARD)"

TYPE_ABBRS = {
    "STREET": ["ST"], "ROAD": ["RD"], "AVENUE": ["AVE", "AV"],
    "CRESCENT": ["CR", "CRES"], "PARADE": ["PDE"], "HIGHWAY": ["HWY", "HWAY"],
    "BOULEVARD": ["BVD", "BLVD"], "COURT": ["CT"], "PLACE": ["PL"],
    "DRIVE": ["DR", "DRV"], "TERRACE": ["TCE", "TERR"], "GROVE": ["GR", "GRV"],
    "LANE": ["LN"], "CIRCUIT": ["CCT"], "CLOSE": ["CL"],
}
STATE_WORDS = ["NEW SOUTH WALES", "VICTORIA", "QUEENSLAND", "TASMANIA",
               "SOUTH AUSTRALIA", "WESTERN AUSTRALIA", "NORTHERN TERRITORY",
               "AUSTRALIAN CAPITAL TERRITORY"]

def _obj(vals):
    """Series that keeps None as None instead of coercing it to float NaN."""
    s = pd.Series(index=P.index, dtype=object)
    s[:] = list(vals)
    return s

def strip_occ(s):
    s = _norm(s)
    for _ in range(4):
        b = s
        s = re.sub(rf"^{NOISE}\s+", "", s)
        s = re.sub(rf"^{CONSUMES}\s*[A-Z]?\d+[A-Z]?\s*/?\s*", "", s)
        s = re.sub(r"^LOT\s*\d+[A-Z]?\s+(?=\d)", "", s)     # lot before a real number
        s = re.sub(r"^[A-Z]?\d+[A-Z]?\s*/\s*", "", s)       # 12/ and G01/
        if s == b: break
    return s

def house_token(s):
    """The written house number, or 'LOT', or None when it cannot be read."""
    t = strip_occ(s)
    m = re.match(r"(\d+[A-Z]?)\s*-\s*(\d+[A-Z]?)", t)
    if m: return f"{m.group(1)}-{m.group(2)}"
    m = re.match(r"(\d+[A-Z]?)", t)
    if m: return m.group(1)
    return "LOT" if re.match(r"^LOT\b", t) else None

def det_prefix(addr):
    """Which one prefix form the writer used. The forms exclude each other,
    as in render(), which makes one roll for all three."""
    s = _norm(addr)
    if re.match(rf"^{CONSUMES}\s*[A-Z]?\d",   s): return "fmt_prefix_unit_kw"
    if re.match(r"^LOT\s*\d+[A-Z]?\s+\d",     s): return "fmt_prefix_lot"
    if re.match(r"^[A-Z]?\d+[A-Z]?\s*/\s*\d", s): return "fmt_prefix_slash"
    return None

def det_state_full(addr):
    t = re.sub(r"\b\d{4}\s*$", "", _norm(addr)).strip()
    return any(t.endswith(w) for w in STATE_WORDS)

def det_abbr_type(addr, type_n):
    w = set(re.findall(r"[A-Z]+", _norm(addr)))
    if type_n in w: return False
    return bool(w & set(TYPE_ABBRS.get(type_n, [])))

def det_abbr_name(addr):
    return bool(set(re.findall(r"[A-Z']+", _norm(addr))) & {"MT", "PT"})

# 5. fmt_ rates, each given eligibility.
h_written = _obj(house_token(a) for a in P.address)
prefix    = _obj(det_prefix(a)  for a in P.address)

# A row whose house number cannot be read is left out of every rate that
# depends on the number. The count and some examples are printed.
readable = h_written.map(lambda t: isinstance(t, str) and t != "LOT")
n_unparsed = int((~readable).sum())
print(f"house number unreadable in {n_unparsed} of {len(P)} matchable rows "
      f"({n_unparsed/max(len(P),1):.1%}); excluded from number-dependent rates")
if n_unparsed:
    for a in P.address[~readable].head(6):
        print(f"    {a}")

k_numeric = P.k.str.fullmatch(r"\d+[A-Za-z]?").fillna(False).astype(bool)
is_range  = ((P.number_last.str.strip() != "")
             & P.k.str.fullmatch(r"\d+").fillna(False)
             & P.number_last.str.fullmatch(r"\d+").fillna(False)).astype(bool)
st_words  = P.street_n.fillna("").str.upper().str.split()

ELIG = {
    # number-form operators need a readable written number to be scored at all
    "fmt_single_vs_range": is_range & readable,
    "fmt_range_written":   (~is_range) & k_numeric & readable,
    "fmt_prefix_slash":    k_numeric,
    "fmt_prefix_unit_kw":  k_numeric,
    "fmt_prefix_lot":      k_numeric,
    "fmt_abbr_type":       P.type_n.fillna("").str.upper().isin(TYPE_ABBRS.keys()),
    # street only, and not rows already flagged err_loc_abbrev (a suburb
    # abbreviation), so one abbreviation is not counted twice
    "fmt_abbr_name":       st_words.map(lambda w: bool({"MOUNT", "PORT"} & set(w)))
                           & ~Fpi.err_loc_abbrev,
    "fmt_abbr_saint":      st_words.map(lambda w: "SAINT" in set(w)),
    "fmt_state_full":      pd.Series(True, index=P.index),
}
FIRED = {
    "fmt_single_vs_range": h_written.map(lambda t: isinstance(t, str) and t != "LOT" and "-" not in t),
    "fmt_range_written":   h_written.map(lambda t: isinstance(t, str) and "-" in t),
    "fmt_prefix_slash":    prefix == "fmt_prefix_slash",
    "fmt_prefix_unit_kw":  prefix == "fmt_prefix_unit_kw",
    "fmt_prefix_lot":      prefix == "fmt_prefix_lot",
    "fmt_abbr_type":       _obj(det_abbr_type(a, t) for a, t
                                in zip(P.address, P.type_n.fillna("").str.upper())),
    "fmt_abbr_name":       _obj(det_abbr_name(a) for a in P.address),
    "fmt_abbr_saint":      pd.Series(False, index=P.index),
    "fmt_state_full":      _obj(det_state_full(a) for a in P.address),
}
FMT_R, FMT_ELIG_N = {}, {}
for k_ in ELIG:
    e = pd.Series(ELIG[k_]).reindex(P.index).fillna(False).astype(bool)
    f = pd.Series(FIRED[k_]).reindex(P.index).fillna(False).astype(bool)
    FMT_ELIG_N[k_] = int(e.sum())
    FMT_R[k_] = float(f[e].mean()) if e.any() else 0.0
# Set to 0 explicitly rather than fitted, because the detector never fires.
FMT_R["fmt_abbr_saint"] = 0.0
P_SINGLE_GIVEN_RANGE = FMT_R["fmt_single_vs_range"]
RANGE_ELIG_CORPUS    = float(is_range.mean())

# 6. Shape of postcode errors, and the mix of negative recipes. After phantom
# numbers, the other two recipes share the rest 55 to 45, a fixed choice.
def _pc(s):
    m = re.findall(r"\b\d{4}\b", str(s)); return m[-1] if m else None
def _pairs(mask):
    out = []
    for a, g in zip(P.address[mask], P.postcode[mask]):
        pa = _pc(a)
        if pa and g: out.append((pa, str(g)))
    return out

_solo = _pairs((Fpi.err_postcode & ~Fpi.err_loc_wrong).to_numpy())
kd = pd.Series([sum(x != y for x, y in zip(a, g)) for a, g in _solo])
PC_K = sorted(kd[kd > 0].unique().tolist())
PC_P = list(np.array([float((kd == k_).sum()) for k_ in PC_K]) /
            max(float((kd > 0).sum()), 1))
P_PC_LEAD = float(np.mean([a[0] != g[0] for a, g in _solo])) if _solo else 0.0

_all = _pairs(Fpi.err_postcode.to_numpy())
_kd  = pd.Series([sum(x != y for x, y in zip(a, g)) for a, g in _all])
PC_TARGET      = _kd[_kd > 0].value_counts(normalize=True).sort_index().round(3).to_dict()
PC_LEAD_TARGET = float(np.mean([a[0] != g[0] for a, g in _all])) if _all else 0.0
PC_ROW_RATE    = float(Fp.err_postcode.sum() / max((n_flags > 0).sum(), 1))

n_phantom = int(F.err_number.sum())
NEG_W = {"phantom_number":     n_phantom / len(neg),
         "street_wrong_place": 0.55 * (1 - n_phantom / len(neg)),
         "street_other_state": 0.45 * (1 - n_phantom / len(neg))}

# 7. Report.
print("\ndraw weights (what the lottery samples) vs flag targets (what cell 10 checks):")
print(pd.DataFrame({"draw_w": pd.Series(ERR_W).round(4),
                    "flag_target": pd.Series(ERR_TARGET).round(4)})
        .sort_values("flag_target", ascending=False).to_string())
print(f"\ndraws per record: { {k_: round(v,4) for k_,v in NDRAW_P.items()} }")
print(f"flags per record: { {k_: round(v,4) for k_,v in FLAG_TARGET_P.items()} }")
print(f"wrong suburb drags postcode: {P_DRAG_PC:.3f}")
print(f"postcode digits differing: { dict(zip(PC_K, np.round(PC_P,3))) }  "
      f"lead-digit rate {P_PC_LEAD:.3f}")
print("\nFORMAT RATES, fitted CONDITIONAL on eligibility (n = eligible rows):")
print(pd.DataFrame({"rate_given_eligible": pd.Series(FMT_R).round(4),
                    "n_eligible": pd.Series(FMT_ELIG_N)}).to_string())
print(f"\nG-NAF match is a range: {RANGE_ELIG_CORPUS*100:.2f}% of the corpus; "
      f"of those, {P_SINGLE_GIVEN_RANGE*100:.1f}% written as a single number")
print("negative recipes:", {k_: round(v, 3) for k_, v in NEG_W.items()})

read 950 rows from output/misspellings.tsv
789 matchable, 161 unmatchable (32 no_record, 129 no_number)


G-NAF components resolved for 789 of 789 matchable rows (100.0%); fmt_ rates are fitted on these
house number unreadable in 12 of 789 matchable rows (1.5%); excluded from number-dependent rates
    Lot 17 Creek Street Riverstone, NSW 2764
    Lot 2506 Bundaleer Street, Belrose NSW 2095
    Lot 2 Cureton Avenue, Mildura, Victoria, 3500
    Shop 1 Morris Arcade Bowra Street Urunga NSW 2455
    Shop 302 Murray St Hobart TAS 7000
    Alkan /126 Racecourse Road EMINGTON VIC 3031

draw weights (what the lottery samples) vs flag targets (what cell 10 checks):
                   draw_w  flag_target
err_postcode       0.2072       0.2400
err_loc_wrong      0.2232       0.2139
err_loc_qualifier  0.1332       0.1277
err_st_edit1       0.0986       0.0946
err_st_type        0.0777       0.0745
err_loc_typo       0.0678       0.0650
err_st_suffix      0.0370       0.0355
err_st_spacing     0.0296       0.0284
err_st_possessive  0.0222       0.0213
err_loc_abbrev     0.0197       0.0189
err_st_edit2

In [ ]:
# Cell 4: the character engine (Kuznetsov and Urdiales), fitted to this
# corpus. The operation split, the position bands and the insertion pool come
# from char_edit_ops.parquet. Substitutions mix the confusion table in
# substitutions.tsv with a category model, in the ratio P_SUB_EMPIRICAL.
# The constants below are used only when those files are missing.

OP_SPLIT = {"insert": 0.388, "delete": 0.354, "substitute": 0.236, "swap": 0.022}
POS_W    = {"start": 0.165, "mid": 0.522, "end": 0.313}
INS_POOL = {"R": 11, "S": 10, "E": 9, "A": 7, "I": 4, "G": 4, "L": 4, "D": 4,
            "Y": 3, "U": 2, "N": 2, "M": 1, "O": 1, "H": 1}

if CHAR_OPS_PQ.exists():
    _o = pd.read_parquet(CHAR_OPS_PQ)
    _c = _o.op.value_counts().to_dict()
    _sub, _swap = _c.get("substitute", 0) - 2 * TRANSPOSE_PAIRS, TRANSPOSE_PAIRS
    _raw = {"insert": _c.get("insert", 0), "delete": _c.get("delete", 0),
            "substitute": max(_sub, 0), "swap": _swap}
    _tot = sum(_raw.values())
    if _tot: OP_SPLIT = {k: v / _tot for k, v in _raw.items()}
    _b = _o.band.value_counts(normalize=True).to_dict()
    if _b: POS_W = {k: _b.get(k, 0.0) for k in ("start", "mid", "end")}
    _ins = _o.loc[_o.op == "insert", "to"].astype(str).str.upper()
    _ins = _ins[_ins.str.fullmatch(r"[A-Z]")].value_counts().to_dict()
    if _ins: INS_POOL = _ins
    print(f"char_edit_ops loaded: {len(_o)} operations")
else:
    print("char_edit_ops.parquet not found, using harvest constants")

_s = sum(POS_W.values()); POS_W = {k: v / _s for k, v in POS_W.items()}
POS_KS, POS_P = list(POS_W), list(POS_W.values())
OP_KS, OP_P = list(OP_SPLIT), list(OP_SPLIT.values())
INS_KS = list(INS_POOL)
INS_P = np.array(list(INS_POOL.values()), float); INS_P /= INS_P.sum()

CONF = {}
if CHAR_SUBS.exists():
    _t = pd.read_csv(CHAR_SUBS, sep="\t")
    _t.columns = [str(c).strip().lower() for c in _t.columns]
    _fc = "from" if "from" in _t.columns else _t.columns[0]
    _tc = "to" if "to" in _t.columns else _t.columns[1]
    _nc = "n" if "n" in _t.columns else ("count" if "count" in _t.columns else _t.columns[2])
    for _, r in _t.iterrows():
        f, t = str(r[_fc]).strip().upper(), str(r[_tc]).strip().upper()
        if not re.fullmatch(r"[A-Z]", f) or not re.fullmatch(r"[A-Z]", t): continue
        CONF.setdefault(f, {})[t] = CONF.setdefault(f, {}).get(t, 0) + float(r[_nc])
    print(f"empirical confusions: {sum(len(v) for v in CONF.values())} pairs "
          f"over {len(CONF)} source characters")
else:
    print("substitutions.tsv not found, category model only")

VOWELS = "AEIOU"
CONSONANTS = "BCDFGHJKLMNPQRSTVWXYZ"
VISUAL = {"L": "I1T", "I": "LJ1", "R": "NP", "N": "MHR", "M": "NW", "Y": "MVX",
          "S": "D58", "D": "SOB", "C": "EGO", "E": "CF", "U": "VY", "V": "UWY",
          "O": "DQ0", "B": "P8R", "G": "CQ6", "W": "VM", "T": "LF", "H": "NK"}
PHONETIC = {"K": "GCQ", "G": "KJ", "C": "KSTQ", "Q": "KC", "P": "BF", "B": "PV",
            "F": "VPH", "V": "FBW", "S": "ZC", "Z": "S", "T": "DC", "D": "TG",
            "J": "GY", "X": "KS", "W": "V", "H": "", "Y": "IJ"}
SUB_CAT_P = {"vowel for vowel": .40, "consonant, other": .20,
             "visual / OCR": .16, "other": .12, "phonetic": .12}
CAT_KS, CAT_P = list(SUB_CAT_P), list(SUB_CAT_P.values())

_emp_hits, _cat_hits = [0], [0]

def _from_category(ch):
    for _ in range(8):
        cat = str(rng.choice(CAT_KS, p=CAT_P))
        if cat == "vowel for vowel":
            pool = [c for c in VOWELS if c != ch] if ch in VOWELS else ""
        elif cat == "visual / OCR":
            pool = [c for c in VISUAL.get(ch, "") if c.isalpha() and c != ch]
        elif cat == "phonetic":
            pool = [c for c in PHONETIC.get(ch, "") if c.isalpha() and c != ch]
        elif cat == "consonant, other":
            pool = [c for c in CONSONANTS if c != ch] if ch in CONSONANTS else ""
        else:
            pool = [c for c in VOWELS + CONSONANTS if c != ch]
        if len(pool): return str(pool[rng.integers(len(pool))])
    pool = [c for c in VOWELS + CONSONANTS if c != ch]
    return str(pool[rng.integers(len(pool))])

def _confusable(ch):
    ch = str(ch).upper()
    if not re.fullmatch(r"[A-Z]", ch): return _from_category("E")
    if ch in CONF and rng.random() < P_SUB_EMPIRICAL:
        ks = list(CONF[ch]); ws = np.array([CONF[ch][k] for k in ks], float)
        _emp_hits[0] += 1
        return str(rng.choice(ks, p=ws / ws.sum()))
    _cat_hits[0] += 1
    return _from_category(ch)

def _insert_char():
    return str(rng.choice(INS_KS, p=INS_P))

def _pos(n):
    for _ in range(8):
        band = str(rng.choice(POS_KS, p=POS_P))
        if   band == "start": lo, hi = (0 if ALLOW_FIRST_CHAR else 1), 2
        elif band == "end":   lo, hi = n - 2, n
        else:                 lo, hi = 2, n - 2
        lo = max(lo, 0 if ALLOW_FIRST_CHAR else 1); hi = min(hi, n)
        if hi > lo: return int(rng.integers(lo, hi))
    return int(rng.integers(1, n))

def char_edit(word, n_edits=1):
    w = str(word)
    if len(w) < 4: return w
    for _ in range(20):
        out = list(w)
        for _ in range(n_edits):
            if len(out) < 4: break
            op = str(rng.choice(OP_KS, p=OP_P))
            i = _pos(len(out))
            if   op == "substitute": out[i] = _confusable(out[i])
            elif op == "insert":     out.insert(i, _insert_char())
            elif op == "delete":     out.pop(i)
            elif op == "swap" and i < len(out) - 1:
                out[i], out[i + 1] = out[i + 1], out[i]
        cand = "".join(out)
        if cand != w and cand.strip() == cand: return cand
    return w

print("op split:", {k: round(v, 3) for k, v in OP_SPLIT.items()})
print("position:", {k: round(v, 3) for k, v in POS_W.items()})
print("demo:", [char_edit("MARRICKVILLE") for _ in range(6)])
print("demo:", [char_edit("FALCON", 2) for _ in range(6)])

char_edit_ops loaded: 195 operations
empirical confusions: 36 pairs over 17 source characters
op split: {'insert': 0.372, 'delete': 0.372, 'substitute': 0.236, 'swap': 0.021}
position: {'start': 0.174, 'mid': 0.518, 'end': 0.308}
demo: ['MARRICKDILLE', 'MARRIKCVILLE', 'MARRSICKVILLE', 'MARRIKVILLE', 'MARRIKCVILLE', 'MARRICVILLE']
demo: ['FLLCON', 'FAGLCRON', 'FALECDON', 'FALON', 'FNICON', 'FAON']


In [ ]:
# Cell 5: the gazetteers, lists of streets, localities and house numbers.
# The error operators in cell 6 use them to pick a real wrong suburb, a real
# other street type, or a house number that does not exist. Reads every
# gnaf_key row into memory and takes about a minute.
C = COLMAP
q = con.execute
st = q(f"""SELECT {C['street']} street, {C['s_type']} s_type,
             {C['suffix']} suffix, {C['locality']} locality,
             {C['state']} state, {C['postcode']} postcode
       FROM {C['streets_table']}""").df()
print(f"{len(st):,} streets loaded")
tf = st.s_type.value_counts(); tf = tf[tf.index.astype(str).str.len() > 0]
TYPES, TYPE_P = list(tf.index), (tf / tf.sum()).to_list()
LOCS = (st.groupby(["locality", "state"]).postcode.agg(lambda s: sorted(set(s))).reset_index())
LOC_BY_STATE = {s: g.reset_index(drop=True) for s, g in LOCS.groupby("state")}
STREETS_IN_LOC = st.groupby("locality").street.agg(set).to_dict()
ALL_STREETS = set(st.street)
nums = q(f"""SELECT {C['postcode']} postcode, {C['street']} street,
              {C['s_type']} s_type, {C['number']} num FROM {C['key_table']}""").df()
# Keyed on street type as well as name, so in a postcode with both Smith
# Street and Smith Lane, the numbers on the Lane do not count as taken on the
# Street.
NUMS_BY_STREET = nums.groupby(["postcode", "street", "s_type"]).num.agg(set).to_dict()
print(f"{len(NUMS_BY_STREET):,} street number sets")
DIRS = ["NORTH", "SOUTH", "EAST", "WEST", "UPPER", "LOWER", "CENTRAL"]
STREETS_BY_STATE = st.groupby("state").street.agg(set).to_dict()

# share of match keys that are a numbered range, printed for reference
RANGE_SHARE = 0.0
if C.get("range_table"):
    try:
        n_rng = q(f"SELECT count(*) FROM {C['range_table']}").fetchone()[0]
        n_key = q(f"SELECT count(*) FROM {C['key_table']}").fetchone()[0]
        RANGE_SHARE = n_rng / max(n_key, 1)
        print(f"{n_rng:,} ranged addresses = {RANGE_SHARE*100:.2f}% of match keys")
    except Exception as e:
        print("range table unavailable, fmt_single_vs_range disabled:", e)

# locality aliases: real colloquial names for gazetted localities, if a table
# is configured
SUBLOCAL = {}
if C.get("alias_table"):
    try:
        al = q(f"SELECT * FROM {C['alias_table']}").df()
        for r in al.itertuples():
            SUBLOCAL.setdefault(str(r.principal_loc).upper(), []).append(str(r.alias_loc).upper())
        print(f"{len(SUBLOCAL):,} localities have a real alias")
    except Exception as e:
        print("alias table unavailable, pattern model only:", e)
else:
    print("no alias table configured, pattern model only")

# tail words that appear at the end of gazetted locality names and that people
# routinely drop ("Avalon Beach" written as "Avalon"), measured off the corpus
TAIL_WORDS = {"BEACH","GROVE","CHASE","HEIGHTS","PARK","POINT","WATERS","VALE",
              "GARDENS","HILL","HILLS","BAY","CREEK","RIDGE","DOWNS","VALLEY",
              "ISLAND","JUNCTION","SPRINGS","FLAT","PLAINS","GULLY","VIEW"}
# tail words people add that are not gazetted ("Thredbo" written "Thredbo Village")
COLLOQ_TAILS = ["VILLAGE","POINT","BEACH","HEIGHTS","GARDENS","CENTRAL","FAIR",
                "CITY","BAY","PARK"]

530,796 streets loaded


490,162 street number sets


1,146,613 ranged addresses = 5.50% of match keys
no alias table configured, pattern model only


In [ ]:
# Cell 6: the error operators, one function per reason in the harvest's list.
# Each operator changes one field and returns the flag or flags it produced,
# or None when it cannot apply and must be redrawn. op_loc_wrong returns two
# flags when it also changes the postcode. op_postcode reproduces the measured
# shape of real postcode errors (cell 3).

def _pick(seq):
    seq = list(seq)
    return seq[rng.integers(len(seq))] if len(seq) else None

def op_st_edit1(r): r["street"] = char_edit(r["street"], 1); return "err_st_edit1"
def op_st_edit2(r): r["street"] = char_edit(r["street"], 2); return "err_st_edit2"
def op_st_heavy(r): r["street"] = char_edit(r["street"], 3); return "err_st_heavy"
def op_st_different(r): r["street"] = char_edit(r["street"], 4); return "err_st_different"

def op_st_possessive(r):
    s = r["street"]
    r["street"] = s[:-1] if s.endswith("S") else s + "S"
    return "err_st_possessive"

def op_st_spacing(r):
    s, w = r["street"], r["street"].split()
    if "'" in s:      r["street"] = s.replace("'", "")
    elif len(w) > 1:  r["street"] = w[0] + w[1] + " " + " ".join(w[2:])
    elif len(s) >= 8:
        i = int(rng.integers(3, len(s) - 3)); r["street"] = s[:i] + " " + s[i:]
    else:             return None
    r["street"] = r["street"].strip()
    return "err_st_spacing"

def op_st_worddrop(r):
    w = r["street"].split()
    if len(w) < 2: return None
    w.pop(int(rng.integers(len(w)))); r["street"] = " ".join(w)
    return "err_st_worddrop"

def op_st_abbrev(r):
    for full, ab in [("MOUNT","MT"), ("SAINT","ST"), ("PORT","PT"),
                     ("NORTH","NTH"), ("SOUTH","STH")]:
        w = r["street"].split()
        if full in w:
            r["street"] = " ".join(ab if t == full else t for t in w)
            return "err_st_abbrev"
    return None

def op_st_type(r):
    for _ in range(10):
        t = str(rng.choice(TYPES, p=TYPE_P))
        if t != r["s_type"]: r["s_type"] = t; return "err_st_type"
    return None

def op_st_suffix(r):
    r["suffix"] = "" if r["suffix"] else str(rng.choice(DIRS[:4]))
    return "err_st_suffix"

def op_loc_wrong(r):
    pool = LOC_BY_STATE.get(r["state"])
    if pool is None or len(pool) < 2: return None
    same_region = pool[pool.postcode.map(
        lambda ps: any(str(p)[:2] == str(r["postcode"])[:2] for p in ps))]
    src = same_region if len(same_region) > 1 and rng.random() < 0.7 else pool
    for _ in range(10):
        row = src.iloc[int(rng.integers(len(src)))]
        if row.locality != r["locality"]:
            r["locality"] = row.locality
            if rng.random() < P_DRAG_PC and len(row.postcode):
                r["postcode"] = str(_pick(row.postcode))
                return "err_loc_wrong;err_postcode"
            return "err_loc_wrong"
    return None

def op_loc_qualifier(r):
    w = r["locality"].split()
    if w[0] in DIRS or w[-1] in DIRS:
        r["locality"] = " ".join(t for t in w if t not in DIRS) or r["locality"]
    else:
        d = str(rng.choice(DIRS[:4]))
        r["locality"] = (r["locality"] + " " + d) if rng.random() < .5 else (d + " " + r["locality"])
    return "err_loc_qualifier"

def op_loc_typo(r):  r["locality"] = char_edit(r["locality"], 1); return "err_loc_typo"
def op_loc_heavy(r): r["locality"] = char_edit(r["locality"], 3); return "err_loc_heavy"

def op_loc_abbrev(r):
    for full, ab in [("MOUNT","MT"), ("SAINT","ST"), ("NORTH","NTH"),
                     ("SOUTH","STH"), ("PORT","PT")]:
        w = r["locality"].split()
        if full in w:
            r["locality"] = " ".join(ab if t == full else t for t in w)
            return "err_loc_abbrev"
    return None

def op_loc_spacing(r):
    w = r["locality"].split()
    if len(w) > 1: r["locality"] = w[0] + w[1] + " " + " ".join(w[2:])
    elif len(r["locality"]) >= 8:
        i = int(rng.integers(3, len(r["locality"]) - 3))
        r["locality"] = r["locality"][:i] + " " + r["locality"][i:]
    else: return None
    r["locality"] = r["locality"].strip()
    return "err_loc_spacing"

def op_loc_sublocal(r):
    """The writer used an unofficial name for the suburb. Three patterns from
    the corpus: drop a gazetted tail word (AVALON BEACH to AVALON), append a
    colloquial one (THREDBO to THREDBO VILLAGE), or use a G-NAF alias. An
    alias is used when one exists, a pattern otherwise."""
    loc = str(r["locality"]).upper()
    pool = SUBLOCAL.get(loc)
    if pool:
        r["locality"] = str(_pick(pool)); return "err_loc_sublocal"
    w = loc.split()
    if len(w) > 1 and w[-1] in TAIL_WORDS and rng.random() < P_SUBLOCAL_DROP:
        r["locality"] = " ".join(w[:-1]); return "err_loc_sublocal"
    if len(w) <= 2 and w[-1] not in COLLOQ_TAILS:
        r["locality"] = loc + " " + str(rng.choice(COLLOQ_TAILS)); return "err_loc_sublocal"
    return None

def op_postcode(r):
    pc = str(r["postcode"]).strip()
    if not re.fullmatch(r"\d{4}", pc): return None
    k = int(rng.choice(PC_K, p=PC_P))
    lead = (k == 4) or (rng.random() < P_PC_LEAD)
    slots = [0] if lead else []
    need = min(k - len(slots), 3)
    if need > 0:
        slots += [int(x) for x in rng.choice([1, 2, 3], size=need, replace=False)]
    slots = sorted(set(slots))
    p = list(pc)
    if (len(slots) == 2 and slots[1] == slots[0] + 1
            and p[slots[0]] != p[slots[1]] and rng.random() < 0.5):
        p[slots[0]], p[slots[1]] = p[slots[1]], p[slots[0]]
    else:
        for i in slots:
            pool = ([d for d in "0234567" if d != p[i]] if i == 0
                    else [d for d in "0123456789" if d != p[i]])
            p[i] = str(pool[rng.integers(len(pool))])
    out = "".join(p)
    if out == pc: return None
    r["postcode"] = out
    return "err_postcode"

def op_state(r):
    states = [s for s in LOC_BY_STATE if s != r["state"]]
    if not states: return None
    r["state"] = str(_pick(states)); return "err_state"

ERR_OPS = {
    "err_st_edit1": op_st_edit1, "err_st_edit2": op_st_edit2,
    "err_st_heavy": op_st_heavy, "err_st_different": op_st_different,
    "err_st_possessive": op_st_possessive, "err_st_spacing": op_st_spacing,
    "err_st_worddrop": op_st_worddrop, "err_st_abbrev": op_st_abbrev,
    "err_st_type": op_st_type, "err_st_suffix": op_st_suffix,
    "err_loc_wrong": op_loc_wrong, "err_loc_qualifier": op_loc_qualifier,
    "err_loc_typo": op_loc_typo, "err_loc_heavy": op_loc_heavy,
    "err_loc_abbrev": op_loc_abbrev, "err_loc_spacing": op_loc_spacing,
    "err_loc_sublocal": op_loc_sublocal,
    "err_postcode": op_postcode, "err_state": op_state,
}
W_NAMES = [k for k in ERR_W if k in ERR_OPS]
W_P = np.array([ERR_W[k] for k in W_NAMES]); W_P /= W_P.sum()
print(f"{len(W_NAMES)} weighted operators ready")

19 weighted operators ready


In [ ]:
# Cell 6b: lookup tables and probabilities used by render().
# P_SAINT, P_NAMEAB and P_SINGLE are the rates fitted in cell 3, the first two
# capped at P_ABBREV_GIVEN_ELIGIBLE. P_INTERIOR, P_RANGE_NOHYPHEN and
# P_RANGE_SLASH are set by hand, because the harvest cannot detect these
# patterns.

TYPE_AB = {full: abbrs[0] for full, abbrs in TYPE_ABBRS.items()}
STATE_FULL = {"NSW":"NEW SOUTH WALES","VIC":"VICTORIA","QLD":"QUEENSLAND",
              "TAS":"TASMANIA","SA":"SOUTH AUSTRALIA","WA":"WESTERN AUSTRALIA",
              "NT":"NORTHERN TERRITORY","ACT":"AUSTRALIAN CAPITAL TERRITORY"}
NAME_AB = {"MOUNT": "MT", "PORT": "PT"}

P_SAINT  = min(P_ABBREV_GIVEN_ELIGIBLE, FMT_R.get("fmt_abbr_saint", 0.0))
P_NAMEAB = min(P_ABBREV_GIVEN_ELIGIBLE, FMT_R.get("fmt_abbr_name",  0.0))
P_SINGLE = min(1.0, FMT_R.get("fmt_single_vs_range", 0.0))
for nm, p, n_ in [("SAINT", P_SAINT, FMT_ELIG_N["fmt_abbr_saint"]),
                  ("MOUNT/PORT", P_NAMEAB, FMT_ELIG_N["fmt_abbr_name"]),
                  ("single-for-range", P_SINGLE, FMT_ELIG_N["fmt_single_vs_range"])]:
    print(f"{nm:18s} p={p:.2f}  fitted on {n_} eligible corpus rows"
          + ("   <- thin, treat as an assumption" if n_ < 15 else ""))

# set by hand, not fitted
P_INTERIOR       = 0.50   # given a span written as one number, how often that
                          # number is not the span's first number
P_RANGE_NOHYPHEN = 0.06   # 46-56 written 4656
P_RANGE_SLASH    = 0.04   # 46-56 written 46/56

for _n in ["TYPE_AB", "STATE_FULL", "NAME_AB", "P_SAINT", "P_NAMEAB",
           "P_SINGLE", "P_INTERIOR", "P_RANGE_NOHYPHEN", "P_RANGE_SLASH"]:
    assert _n in globals(), f"{_n} missing; render() in cell 7 will fail"
print("renderer knobs ready")

SAINT              p=0.00  fitted on 0 eligible corpus rows   <- thin, treat as an assumption
MOUNT/PORT         p=0.67  fitted on 3 eligible corpus rows   <- thin, treat as an assumption
single-for-range   p=0.71  fitted on 154 eligible corpus rows
renderer knobs ready


In [ ]:
# Cell 7: render(), which turns the components into one uppercase string
# with no commas and applies the format operations.
# Returns (string, fmt_ tags applied, elig_ tags). An elig_ tag records that a
# format operation could have fired on this record, so cell 10 can compare
# rates on eligible rows only.

def render(r):
    ops, elig, roll = [], [], rng.random
    num  = str(r["number"]).upper().strip()
    last = str(r.get("number_last", "")).upper().strip()
    has_range = bool(last) and num[:1].isdigit()

    # number: range or single, interior number, range separator
    if has_range:
        elig.append("elig_single_vs_range")
        if roll() < P_SINGLE:
            ops.append("fmt_single_vs_range")
            # one number of the span ("94" for 64-106, or "50" for 46-50):
            # only pick one with no G-NAF record of its own, keeping the
            # endpoint's odd/even parity.
            if roll() < P_INTERIOR:
                try:
                    lo = int(re.match(r"\d+", num).group())
                    hi = int(re.match(r"\d+", last).group())
                    taken = NUMS_BY_STREET.get((r["postcode"], r["street"], r["s_type"]), set())
                    # hi + 1 so the last number of the span can be written
                    # ("124 Hassell St" for 122-124)
                    cands = [n for n in range(lo + 1, hi + 1)
                             if n % 2 == lo % 2
                             and n not in taken and str(n) not in taken]
                    if cands:
                        num = str(cands[int(rng.integers(len(cands)))])
                        ops.append("fmt_interior_number")
                except Exception:
                    pass
        else:
            num = f"{num}-{last}"

    if num and num[0].isdigit():
        elig.append("elig_prefix")
        if not has_range:
            elig.append("elig_range_written")
            if roll() < FMT_R.get("fmt_range_written", 0):
                try:
                    n0 = int(re.match(r"\d+", num).group())
                    num = f"{n0}-{n0 + 2 + 2 * int(rng.integers(3))}"
                    ops.append("fmt_range_written")
                except Exception:
                    pass

        # change the range separator before the prefix, so "46-56" written
        # "4656" can still gain a unit prefix and become "8/4656"
        if re.fullmatch(r"\d+-\d+", num):
            pr2 = roll()
            if pr2 < P_RANGE_NOHYPHEN:
                num = num.replace("-", "");  ops.append("fmt_range_nohyphen")
            elif pr2 < P_RANGE_NOHYPHEN + P_RANGE_SLASH:
                num = num.replace("-", "/"); ops.append("fmt_range_slash")

        pr = roll()
        p_slash = FMT_R.get("fmt_prefix_slash", 0)
        p_unit  = FMT_R.get("fmt_prefix_unit_kw", 0)
        p_lot   = FMT_R.get("fmt_prefix_lot", 0)
        if pr < p_slash:
            num = f"{1 + int(rng.integers(30))}/{num}"; ops.append("fmt_prefix_slash")
        elif pr < p_slash + p_unit:
            kw = str(rng.choice(["UNIT","LEVEL","SHOP","SUITE"], p=[.55,.25,.12,.08]))
            num = f"{kw} {1 + int(rng.integers(30))} {num}"; ops.append("fmt_prefix_unit_kw")
        elif pr < p_slash + p_unit + p_lot:
            # the lot is added before the house number, never in place of it,
            # as in the corpus ("Lot 36, 146 Parramatta Road Homebush"). The
            # lot number itself is made up.
            num = f"LOT {1 + int(rng.integers(900))} {num}"
            ops.append("fmt_prefix_lot")

    # street: SAINT, MOUNT and PORT abbreviations
    street = str(r["street"]).upper()
    w = street.split()
    if "SAINT" in w:
        elig.append("elig_abbr_saint")
        if roll() < P_SAINT:
            street = " ".join("ST" if t == "SAINT" else t for t in w)
            ops.append("fmt_abbr_saint")
    else:
        for full, ab in NAME_AB.items():                 # MOUNT->MT, PORT->PT
            if full in w:
                elig.append("elig_abbr_name")
                if roll() < P_NAMEAB:
                    street = " ".join(ab if t == full else t for t in street.split())
                    ops.append("fmt_abbr_name")
                break

    # street type: abbreviate (ST, RD, AVE, ...)
    s_type = str(r["s_type"]).upper().strip()
    if s_type in TYPE_AB:
        elig.append("elig_abbr_type")
        if roll() < FMT_R.get("fmt_abbr_type", 0):
            s_type = TYPE_AB[s_type]
            ops.append("fmt_abbr_type")

    # state: full name instead of abbreviation
    state = str(r["state"]).upper().strip()
    if state in STATE_FULL:
        elig.append("elig_state_full")
        if roll() < FMT_R.get("fmt_state_full", 0):
            state = STATE_FULL[state]
            ops.append("fmt_state_full")

    # suffix, locality, postcode, then join
    suffix   = str(r.get("suffix", "")).upper().strip()
    locality = str(r["locality"]).upper().strip()
    postcode = str(r["postcode"]).upper().strip()

    s = " ".join(t for t in [num, street, s_type, suffix, locality, state, postcode] if t)
    s = re.sub(r"\s+", " ", s).strip()
    return s, ops, elig

In [ ]:
# Cell 8: the generators for mutated positives, clean positives and negatives.
# OP_TRIED and OP_APPLIED count how often each error operator was drawn and how
# often it applied. An operator that needs something the record lacks (a SAINT
# to abbreviate) returns None and is redrawn, so cell 10 reports a refusal
# rate beside each operator.

from collections import Counter
ND_KS = list(NDRAW_P)
ND_P = np.array(list(NDRAW_P.values()), float); ND_P /= ND_P.sum()
OP_TRIED, OP_APPLIED = Counter(), Counter()

def clean_string(r):
    num = str(r["number"])
    last = str(r.get("number_last", "") or "")
    if last and num[:1].isdigit(): num = f"{num}-{last}"
    return " ".join(t for t in [num, r["street"], r["s_type"], r["suffix"],
                                r["locality"], r["state"], str(r["postcode"])] if t)

def make_positive(row):
    r = dict(row); anchor = clean_string(r)
    n = int(rng.choice(ND_KS, p=ND_P))
    flags, tried, applied = [], set(), 0
    while applied < n and len(tried) < len(W_NAMES):
        name = str(rng.choice(W_NAMES, p=W_P))
        if name in tried: continue
        tried.add(name)
        OP_TRIED[name] += 1
        got = ERR_OPS[name](r)
        if got:
            OP_APPLIED[name] += 1
            flags.extend(got.split(";")); applied += 1
    if not flags: return None
    s, fmt, elig = render(r)
    return dict(dirty=s, anchor=anchor, label="POS",
                ops=";".join(flags + fmt), elig=";".join(elig))

def make_clean(row):
    r = dict(row); anchor = clean_string(r)
    s, fmt, elig = render(r)
    return dict(dirty=s, anchor=anchor, label="POS_CLEAN",
                ops=";".join(fmt), elig=";".join(elig))

DROPS = {}
def make_negative(row):
    r = dict(row); anchor = clean_string(r)
    kind = str(rng.choice(list(NEG_W), p=list(NEG_W.values())))
    if kind == "phantom_number":
        have = NUMS_BY_STREET.get((r["postcode"], r["street"], r["s_type"]), set())
        was_lot = str(r["number"]).upper().strip().startswith("LOT")
        for _ in range(30):
            cand = f"LOT {int(rng.integers(1, 900))}" if was_lot else str(int(rng.integers(1, 400)))
            if cand not in have and cand != str(r["number"]):
                r["number"], r["number_last"] = cand, ""; break
        else:
            DROPS[kind] = DROPS.get(kind, 0) + 1; return None
    elif kind == "street_wrong_place":
        here = STREETS_IN_LOC.get(r["locality"], set())
        pool = list(ALL_STREETS)
        for _ in range(30):
            other = str(pool[rng.integers(len(pool))])
            if other not in here and other != r["street"]:
                r["street"] = other; break
        else:
            DROPS[kind] = DROPS.get(kind, 0) + 1; return None
    else:
        home = STREETS_BY_STATE.get(r["state"], set())
        others = [s for s in STREETS_BY_STATE if s != r["state"]]
        if not others: return None
        for _ in range(30):
            pool = list(STREETS_BY_STATE[str(rng.choice(others))])
            if not pool: continue
            cand = str(pool[rng.integers(len(pool))])
            if cand not in home:
                r["street"] = cand; break
        else:
            DROPS[kind] = DROPS.get(kind, 0) + 1; return None
    s, fmt, elig = render(r)
    return dict(dirty=s, anchor=anchor, label=f"NEG_{kind.upper()}",
                ops=";".join([kind] + fmt), elig=";".join(elig))

In [ ]:
# Cell 8b: a second sampler that draws only records with a numbered range.
# Ranges are 5.5% of G-NAF match keys (cell 5) but about 20% of the test set's
# matchable rows (cell 3), so cell 9 oversamples them. No fitted rate changes.
# The WHERE clause keeps records whose key is number_first and whose two ends
# are numeric, so each one renders as a "first-last" anchor.

RANGE_OVERSAMPLE = 0.30   # share of batches drawn from the range pool instead
                          # of all of G-NAF; 0.0 turns oversampling off

C = COLMAP
if C.get("range_table"):
    RANGE_SQL_TEMPLATE = f"""
    SELECT k.{C['number']} AS number, g.number_last AS number_last,
           k.{C['street']} AS street,
           k.{C['s_type']} AS s_type, coalesce(k.{C['suffix']}, '') AS suffix,
           k.{C['locality']} AS locality, k.{C['state']} AS state,
           k.{C['postcode']} AS postcode
    FROM {C['key_table']} k
    JOIN {C['range_table']} g ON g.pid = k.{C['pid']}
    WHERE k.{C['number']} = g.number_first
      AND regexp_matches(k.{C['number']}, '^[0-9]+$')
      AND regexp_matches(g.number_last, '^[0-9]+$')
    USING SAMPLE reservoir({BATCH} ROWS) REPEATABLE ({{seed}})
    """
    _probe = con.execute(RANGE_SQL_TEMPLATE.format(seed=SEED)).df()
    _with_last = (_probe.number_last.astype(str).str.strip() != "").mean() if len(_probe) else 0.0
    print(f"range pool sample: {len(_probe):,} rows, "
          f"{_with_last*100:.1f}% carry a numeric number_last (want ~100%)")
    assert len(_probe) > 0 and _with_last > 0.98, \
        "range sampler returned no clean ranges; check gnaf_range and the k = number_first join"
else:
    RANGE_SQL_TEMPLATE = None
    RANGE_OVERSAMPLE = 0.0
    print("no range_table configured; RANGE_OVERSAMPLE forced to 0.0")

range pool sample: 3,960 rows, 100.0% carry a numeric number_last (want ~100%)


In [ ]:
# Cell 8c: the exclusion set, every test-set answer rendered as an anchor
# string. Cell 9 skips any sampled record whose anchor is in this set, so no
# test answer appears in training. Only the exact record is excluded, not its
# street, because the retrieval database holds every G-NAF record.

MISS_TSV = next((p for p in [EXPORT_IN / "misspellings.tsv",
                             Path("misspellings.tsv")] if p.exists()), None)
assert MISS_TSV is not None, \
    "misspellings.tsv not found; cannot build the exclusion set"

def _norm_anchor(s):
    """One form for comparing anchor strings: uppercase, single spaces,
    trimmed. Both sides of every comparison go through it."""
    return re.sub(r"\s+", " ", str(s).upper()).strip()

_miss = pd.read_csv(MISS_TSV, sep="\t", dtype=str).fillna("")
# `matched` holds the raw G-NAF ADDRESS_LABEL ("125 PARK RD, ..."), so it is
# normalised the same way the join below normalises ADDRESS_LABEL.
_miss["m"] = _miss.matched.map(
    lambda s: re.sub(r"\s+", " ", str(s).upper().replace(",", " ")).strip())
_miss = _miss[_miss.m != ""]
print(f"{len(_miss):,} labelled rows read from {MISS_TSV.name}")

# Render each label's gnaf_key rows the way the synthesiser renders an anchor.
# One label can match many pids (every unit in a tower) and one pid can hold
# two keys (street number and LOT form). Every form goes into the set.
con.register("miss_df", _miss[["m"]].drop_duplicates())
_ta = con.sql(r"""
    SELECT DISTINCT
           regexp_replace(trim(concat_ws(' ',
               CASE WHEN g.number_last IS NOT NULL AND g.number_last <> ''
                         AND regexp_matches(k.k, '^[0-9]')
                    THEN k.k || '-' || g.number_last ELSE k.k END,
               k.street_n, k.type_n, k.suffix_n,
               k.locality_n, k.state_n, k.postcode)),
               '\s+', ' ', 'g') AS anchor
    FROM miss_df m
    JOIN gnaf_key k
      ON regexp_replace(trim(replace(upper(k.ADDRESS_LABEL), ',', ' ')),
                        '\s+', ' ', 'g') = m.m
    LEFT JOIN gnaf_range g ON g.pid = k.pid AND g.number_first = k.k
""").df()

TEST_ANCHORS = set(_ta.anchor.map(_norm_anchor))
print(f"exclusion set: {len(TEST_ANCHORS):,} distinct anchor strings")
assert len(TEST_ANCHORS) > 500, \
    ("exclusion set is suspiciously small; the ADDRESS_LABEL join is probably "
     "failing. Check that misspellings.tsv's `matched` column still holds raw "
     "G-NAF labels rather than rendered strings.")

print("\nsample excluded anchors:")
for a in sorted(TEST_ANCHORS)[:5]:
    print(f"  {a}")

# If a synthetic file already exists, count its anchors that are in the set.
# This checks the set against the file written by an earlier run of cell 9.
_syn_path = OUT / "synthetic_addresses.parquet"
if _syn_path.exists():
    _old = set(pd.read_parquet(_syn_path).query("label == 'POS'")
                 .anchor.map(_norm_anchor))
    print(f"\ncollisions with the existing synthetic file: "
          f"{len(TEST_ANCHORS & _old)}")
    print("  (if it is 0, the set "
          "is not built the same way cell 9 builds anchors)")

945 labelled rows read from misspellings.tsv


exclusion set: 1,139 distinct anchor strings

sample excluded anchors:
  1 ALFRED STREET SYDNEY NSW 2000
  1 AMAROO GROVE BURNSIDE VIC 3023
  1 ANZAC STREET NORTH ST MARYS NSW 2760
  1 CASTLE HILL ROAD WEST PENNANT HILLS NSW 2125
  1 CHIDLEY WAY MOSMAN PARK WA 6012



collisions with the existing synthetic file: 4
  (if it is 0, the set is not built the same way cell 9 builds anchors)


In [ ]:
# Cell 9: the main loop, TARGET_N mutated rows then N_CLEAN clean rows.
# Each batch is a seeded DuckDB sample, so the run reproduces from SEED.
# Records whose anchor is a test answer (cell 8c) are skipped before any
# operator is applied. The rows are shuffled, then written to
# output_synthesis/synthetic_addresses.parquet and sample_500.tsv.

C = COLMAP
RANGE_JOIN = (f"LEFT JOIN {C['range_table']} g ON g.pid = k.{C['pid']}"
              if C.get("range_table") else "")
RANGE_COL = (f"CASE WHEN k.{C['number']} = g.number_first THEN coalesce(g.number_last,'') "
             f"ELSE '' END" if C.get("range_table") else "''")

# Every component comes from the same gnaf_key row, so each sampled record is
# a real address. A join to gnaf_streets on postcode and street name would pair
# a number with the wrong street type wherever one postcode has both Smith
# Street and Smith Lane.
SQL_TEMPLATE = f"""
SELECT k.{C['number']} AS number, {RANGE_COL} AS number_last,
       k.{C['street']} AS street,
       k.{C['s_type']} AS s_type, coalesce(k.{C['suffix']}, '') AS suffix,
       k.{C['locality']} AS locality, k.{C['state']} AS state,
       k.{C['postcode']} AS postcode
FROM {C['key_table']} k
{RANGE_JOIN}
USING SAMPLE reservoir({BATCH} ROWS) REPEATABLE ({{seed}})
"""

# Guard: the FROM and JOIN clause must return exactly one row per gnaf_key row.
_n_key  = con.execute(f"SELECT count(*) FROM {C['key_table']}").fetchone()[0]
_n_join = con.execute(
    f"SELECT count(*) FROM {C['key_table']} k {RANGE_JOIN}").fetchone()[0]
assert _n_join == _n_key, (
    f"chimera regression: the FROM/JOIN clause returns {_n_join:,} rows for "
    f"{_n_key:,} gnaf_key rows -- something is fanning out. Check RANGE_JOIN "
    f"(gnaf_range should be at most one row per pid) before sampling.")
print(f"chimera guard OK: {_n_key:,} gnaf_key rows, {_n_join:,} after the join "
      f"(1:1, no fan-out)")

assert "TEST_ANCHORS" in globals(), "run cell 8c first; TEST_ANCHORS is missing"
OP_TRIED.clear(); OP_APPLIED.clear(); DROPS.clear()
print(f"test-answer exclusion active: {len(TEST_ANCHORS):,} anchors blocked")

out, seen, batch_i, dropped, skipped_test = [], set(), 0, 0, 0
while len(out) < TARGET_N:
    # draw this batch from the range pool RANGE_OVERSAMPLE of the time, from
    # all of G-NAF otherwise; the two seed streams are offset so they never meet
    if rng.random() < RANGE_OVERSAMPLE:
        batch = con.execute(RANGE_SQL_TEMPLATE.format(seed=SEED + 500_000 + batch_i)).df().fillna("")
    else:
        batch = con.execute(SQL_TEMPLATE.format(seed=SEED + batch_i)).df().fillna("")
    batch_i += 1
    for row in batch.to_dict("records"):
        if not str(row["street"]).strip(): continue
        # skip any record that answers a test query; make_positive and
        # make_negative build the anchor with the same clean_string
        if _norm_anchor(clean_string(row)) in TEST_ANCHORS:
            skipped_test += 1
            continue
        rec = make_negative(row) if rng.random() < NEG_FRAC else make_positive(row)
        if rec is None: dropped += 1; continue
        if rec["dirty"] in seen: continue
        seen.add(rec["dirty"]); out.append(rec)
        if len(out) >= TARGET_N: break
    print(f"  {len(out):,} / {TARGET_N:,}  (batch {batch_i}, dropped {dropped} "
          f"{DROPS}, test-answers skipped {skipped_test})")
    if batch_i > 200: print("  stopping: not enough distinct records"); break

n_clean = 0
while n_clean < N_CLEAN:
    batch = con.execute(SQL_TEMPLATE.format(seed=SEED + 10_000 + batch_i)).df().fillna("")
    batch_i += 1
    for row in batch.to_dict("records"):
        if not str(row["street"]).strip(): continue
        if _norm_anchor(clean_string(row)) in TEST_ANCHORS:
            skipped_test += 1
            continue
        rec = make_clean(row)
        if rec["dirty"] in seen: continue
        seen.add(rec["dirty"]); out.append(rec); n_clean += 1
        if n_clean >= N_CLEAN: break
    print(f"  clean {n_clean:,} / {N_CLEAN:,}")

print(f"\ntest answers refused during sampling: {skipped_test:,}")

df = pd.DataFrame(out).sample(frac=1, random_state=SEED).reset_index(drop=True)
df.to_parquet(OUT / "synthetic_addresses.parquet", index=False)
df.sample(min(500, len(df)), random_state=SEED).to_csv(OUT / "sample_500.tsv", sep="\t", index=False)
print(df.label.value_counts().to_string())

chimera guard OK: 20,837,941 gnaf_key rows, 20,837,941 after the join (1:1, no fan-out)
test-answer exclusion active: 1,139 anchors blocked


  4,995 / 100,000  (batch 1, dropped 0 {}, test-answers skipped 5)


  9,991 / 100,000  (batch 2, dropped 0 {}, test-answers skipped 9)


  14,987 / 100,000  (batch 3, dropped 0 {}, test-answers skipped 13)


  19,983 / 100,000  (batch 4, dropped 0 {}, test-answers skipped 17)


...

  83,261 / 100,000  (batch 18, dropped 1 {'phantom_number': 1}, test-answers skipped 150)


  88,257 / 100,000  (batch 19, dropped 1 {'phantom_number': 1}, test-answers skipped 152)


  93,252 / 100,000  (batch 20, dropped 1 {'phantom_number': 1}, test-answers skipped 156)


  98,248 / 100,000  (batch 21, dropped 1 {'phantom_number': 1}, test-answers skipped 160)


  100,000 / 100,000  (batch 22, dropped 1 {'phantom_number': 1}, test-answers skipped 171)

test answers refused during sampling: 171
label
POS                       84971
NEG_PHANTOM_NUMBER        11936
NEG_STREET_WRONG_PLACE     1720
NEG_STREET_OTHER_STATE     1373


In [ ]:
# Cell 10: validation, the synthetic rates against the rates fitted in cell 3.
# The error table adds a `refused` column (drawn but not applicable, so
# redrawn). The format table uses render()'s elig_ tags and covers POS rows
# only, the population the rates were fitted on.

pos_m = df[df.label == "POS"]

op_counts = pd.Series([o for ops in pos_m.ops for o in ops.split(";")
                       if o.startswith("err_")]).value_counts()
cmp = pd.DataFrame({"synthetic": (op_counts / op_counts.sum()).round(3),
                    "target": pd.Series(ERR_TARGET).round(3)}).fillna(0)
cmp["gap"] = (cmp.synthetic - cmp.target).round(3)
cmp["drawn"]   = pd.Series(dict(OP_TRIED)).reindex(cmp.index).fillna(0).astype(int)
cmp["refused"] = (1 - pd.Series(dict(OP_APPLIED)).reindex(cmp.index).fillna(0)
                      / cmp.drawn.replace(0, np.nan)).round(3)
print("ERROR OPERATORS (share of all error flags)")
print("  `refused` = drawn but inapplicable to the sampled record, so redrawn.")
print("  A negative gap with a high refusal rate is eligibility, not a bug.")
print(cmp.sort_values("target", ascending=False).to_string(), "\n")

n_per = pos_m.ops.str.count("err_")
print("FLAGS PER MUTATED RECORD")
print(pd.DataFrame({"synthetic": n_per.value_counts(normalize=True).sort_index().round(3),
                    "target": pd.Series(FLAG_TARGET_P).round(3)}).fillna(0).to_string(), "\n")

# format operators, using render()'s own eligibility tags
ELIG_FOR = {"fmt_single_vs_range": "elig_single_vs_range",
            "fmt_range_written":   "elig_range_written",
            "fmt_prefix_slash":    "elig_prefix",
            "fmt_prefix_unit_kw":  "elig_prefix",
            "fmt_prefix_lot":      "elig_prefix",
            "fmt_abbr_saint":      "elig_abbr_saint",
            "fmt_abbr_name":       "elig_abbr_name",
            "fmt_abbr_type":       "elig_abbr_type",
            "fmt_state_full":      "elig_state_full"}
_ops  = pos_m.ops.fillna("").str.split(";").map(set)
_elig = pos_m.elig.fillna("").str.split(";").map(set)
rows = []
for fmt_k, elig_k in ELIG_FOR.items():
    e = _elig.map(lambda s, t=elig_k: t in s)
    f = _ops.map(lambda s, t=fmt_k: t in s)
    rows.append({"target_given_eligible": round(FMT_R.get(fmt_k, 0), 4),
                 "realised_given_eligible": round(float(f[e].mean()) if e.any() else 0.0, 4),
                 "eligible_share": round(float(e.mean()), 4),
                 "n_eligible": int(e.sum()),
                 "realised_overall": round(float(f.mean()), 4)})
fmt_tab = pd.DataFrame(rows, index=list(ELIG_FOR))
fmt_tab["gap"] = (fmt_tab.realised_given_eligible - fmt_tab.target_given_eligible).round(4)
print("FORMAT OPERATORS (POS rows only; rates are conditional on eligibility)")
print(fmt_tab.to_string(), "\n")
_bad = fmt_tab[fmt_tab.gap.abs() > 0.02]
if len(_bad):
    print("  CHECK: these exceed a 0.02 conditional gap:", list(_bad.index), "\n")

# the three hand-set number operators
print("HAND-SET NUMBER OPERATORS (declared, not fitted)")
_rng_elig = _elig.map(lambda s: "elig_single_vs_range" in s)
for k_, knob, base in [("fmt_interior_number", P_INTERIOR, "fmt_single_vs_range"),
                       ("fmt_range_nohyphen", P_RANGE_NOHYPHEN, None),
                       ("fmt_range_slash",    P_RANGE_SLASH,    None)]:
    fired = _ops.map(lambda s, t=k_: t in s)
    if base:
        b = _ops.map(lambda s, t=base: t in s)
        cond = float(fired[b].mean()) if b.any() else 0.0
        print(f"  {k_:20s} {int(fired.sum()):>6,} rows   given {base}: "
              f"{cond:.3f}   knob {knob:.2f}")
    else:
        print(f"  {k_:20s} {int(fired.sum()):>6,} rows   overall "
              f"{fired.mean()*100:.2f}%   knob {knob:.2f}")
print(f"  range anchors: {_rng_elig.mean()*100:.1f}% of POS "
      f"(corpus base rate {RANGE_ELIG_CORPUS*100:.1f}%; the difference is "
      f"RANGE_OVERSAMPLE={RANGE_OVERSAMPLE}, a distribution correction)\n")

# postcode error shape, lot numbers, character engine
def _pcx(s):
    m = re.findall(r"\b\d{4}\b", str(s)); return m[-1] if m else None
pcp = [(_pcx(a), _pcx(b)) for a, b in zip(pos_m.dirty, pos_m.anchor)]
pcp = [(a, b) for a, b in pcp if a and b and a != b]
if pcp:
    lead = np.mean([a[0] != b[0] for a, b in pcp])
    kdx = pd.Series([sum(x != y for x, y in zip(a, b)) for a, b in pcp])
    print("POSTCODE ERROR SHAPE")
    print(f"  rows with a changed postcode : {len(pcp)/len(pos_m)*100:.1f}% of POS rows"
          f"   target {PC_ROW_RATE*100:.1f}%")
    print(f"  leading digit changed        : {lead*100:.1f}%   target {PC_LEAD_TARGET*100:.1f}%")
    print(f"  digits differing             : {dict(kdx.value_counts(normalize=True).sort_index().round(3))}"
          f"   target {PC_TARGET}\n")

lot_a = df.anchor.str.upper().str.startswith("LOT")
lot_d = df.dirty.str.upper().str.contains(r"\bLOT\b")
print("LOT-NUMBER BOOKKEEPING")
print(f"  anchors with a lot number    : {lot_a.mean()*100:5.2f}%")
print(f"  dirty  with a lot number     : {lot_d.mean()*100:5.2f}%")
print(f"  lot ADDED  (not in anchor)   : {(lot_d & ~lot_a).mean()*100:5.2f}%"
      f"   target {FMT_R.get('fmt_prefix_lot',0)*100:.2f}% given eligible")
print(f"  lot DROPPED (was in anchor)  : {(~lot_d & lot_a).mean()*100:5.2f}%   target 0.00%\n")

tot = _emp_hits[0] + _cat_hits[0]
print(f"CHARACTER ENGINE: {_emp_hits[0]:,} substitutions from the empirical table, "
      f"{_cat_hits[0]:,} from the category model "
      f"({_emp_hits[0]/max(tot,1)*100:.1f}% empirical, knob is {P_SUB_EMPIRICAL:.0%})")
print("  NOTE: char_edit_ops.parquet and substitutions.tsv are still harvest\n"
      "  outputs fitted before the label corrections. Character statistics are\n"
      "  barely affected by 23 relabelled rows. Every other rate is fitted to\n"
      "  the current file.")
print(f"RECORDS DROPPED as unusable: {dropped:,}  by recipe: {DROPS}")
print("\nSample rows to check by eye: output_synthesis/sample_500.tsv")

ERROR OPERATORS (share of all error flags)
  `refused` = drawn but inapplicable to the sampled record, so redrawn.
  A negative gap with a high refusal rate is eligibility, not a bug.
                   synthetic  target    gap  drawn  refused
err_postcode           0.251   0.240  0.011  21122    0.000
err_loc_wrong          0.224   0.214  0.010  22878    0.000
err_loc_qualifier      0.136   0.128  0.008  13919    0.000
err_st_edit1           0.100   0.095  0.005  10229    0.000
err_st_type            0.081   0.074  0.007   8214    0.000
err_loc_typo           0.070   0.065  0.005   7171    0.000
err_st_suffix          0.038   0.035  0.003   3926    0.000
err_st_spacing         0.013   0.028 -0.015   3081    0.582
err_st_possessive      0.023   0.021  0.002   2354    0.000
err_loc_abbrev         0.002   0.019 -0.017   2074    0.924
err_st_edit2           0.020   0.019  0.001   2069    0.000
err_loc_sublocal       0.018   0.018  0.000   1921    0.060
err_st_worddrop        0.001   0.015

FORMAT OPERATORS (POS rows only; rates are conditional on eligibility)
                     target_given_eligible  realised_given_eligible  eligible_share  n_eligible  realised_overall     gap
fmt_single_vs_range                 0.7078                   0.7066          0.2927       24869            0.2068 -0.0012
fmt_range_written                   0.0353                   0.0346          0.5194       44135            0.0179 -0.0007
fmt_prefix_slash                    0.0825                   0.0820          0.8121       69004            0.0666 -0.0005
fmt_prefix_unit_kw                  0.1142                   0.1150          0.8121       69004            0.0934  0.0008
fmt_prefix_lot                      0.0013                   0.0011          0.8121       69004            0.0009 -0.0002
fmt_abbr_saint                      0.0000                   0.0000          0.0002          19            0.0000  0.0000
fmt_abbr_name                       0.6667                   0.7126        

POSTCODE ERROR SHAPE
  rows with a changed postcode : 29.6% of POS rows   target 28.8%
  leading digit changed        : 7.1%   target 5.9%
  digits differing             : {1: np.float64(0.54), 2: np.float64(0.319), 3: np.float64(0.121), 4: np.float64(0.02)}   target {1: 0.616, 2: 0.266, 3: 0.099, 4: 0.02}

LOT-NUMBER BOOKKEEPING
  anchors with a lot number    : 18.74%
  dirty  with a lot number     : 18.82%
  lot ADDED  (not in anchor)   :  0.08%   target 0.13% given eligible
  lot DROPPED (was in anchor)  :  0.00%   target 0.00%

CHARACTER ENGINE: 4,116 substitutions from the empirical table, 1,700 from the category model (70.8% empirical, knob is 80%)
  NOTE: char_edit_ops.parquet and substitutions.tsv are still harvest
  outputs fitted before the label corrections. Character statistics are
  barely affected by 23 relabelled rows. Every other rate is fitted to
  the current file.
RECORDS DROPPED as unusable: 1  by recipe: {'phantom_number': 1}

Sample rows to check by eye: output_sy

In [ ]:
# Cell 10b: check the range anchors and the number written for a span in the
# synthetic file from cell 9. The house number is the first number after any
# UNIT, LEVEL, SHOP, SUITE or LOT prefix and any slash unit.

import pandas as pd, re
from pathlib import Path

SYN = next((Path(p) for p in ["output_synthesis/synthetic_addresses.parquet",
                              "output_synthesis/sample_500.tsv"] if Path(p).exists()), None)
assert SYN is not None, "no synthetic output found; regenerate first"
syn = pd.read_parquet(SYN) if SYN.suffix == ".parquet" else pd.read_csv(SYN, sep="\t", dtype=str)
syn = syn.fillna("")
pos = syn[syn.label == "POS"] if "label" in syn.columns else syn
print(f"read {len(pos):,} POS rows from {SYN.name}")

def anchor_is_range(a):
    return bool(re.match(r"^(?:LOT\s+)?\d+-\d+\b", str(a).upper().strip()))

rng_rows = pos[pos.anchor.map(anchor_is_range)]
print(f"anchors that are range records: {len(rng_rows):,} "
      f"({len(rng_rows)/max(len(pos),1)*100:.1f}% of POS)\n")
print("sample range anchors (anchor should be first-last form):")
print(rng_rows[["dirty", "anchor"]].head(8).to_string(index=False))

def house_number(dirty):
    s = str(dirty).upper().strip()
    s = re.sub(r"^(?:UNIT|LEVEL|SHOP|SUITE|LOT)\s+\d+[A-Z]?\s+", "", s)   # keyword unit
    s = re.sub(r"^\d+[A-Z]?\s*/\s*", "", s)                              # slash unit
    m = re.match(r"(\d+)", s)
    return int(m.group(1)) if m else None

if "ops" in pos.columns:
    inter = pos[pos.ops.str.contains("fmt_interior_number", regex=False)].copy()
    def classify(row):
        n = house_number(row.dirty)
        rm = re.match(r"^(?:LOT\s+)?(\d+)-(\d+)", str(row.anchor).upper().strip())
        if n is None or not rm: return "unparsed"
        lo, hi = int(rm.group(1)), int(rm.group(2))
        if n == lo:      return "equals_first"     # the first number, not interior
        if lo < n <= hi: return "true_interior"    # a middle number or the last
        return "outside"                           # outside the span
    inter["klass"] = inter.apply(classify, axis=1)
    print(f"\nfmt_interior_number rows: {len(inter):,}")
    print(inter.klass.value_counts(normalize=True).round(3).to_string())
    print("\nsample TRUE interior positives:")
    print(inter[inter.klass == "true_interior"][["dirty", "anchor"]].head(6).to_string(index=False))
    bad = inter[inter.klass == "outside"]
    if len(bad):
        print(f"\n{len(bad)} rows landed OUTSIDE the range (should be ~0):")
        print(bad[["dirty", "anchor"]].head(6).to_string(index=False))

read 84,971 POS rows from synthetic_addresses.parquet
anchors that are range records: 24,835 (29.2% of POS)

sample range anchors (anchor should be first-last form):
                                          dirty                                      anchor
              204 STURT STREET ADELAIDE SA 5120       204-206 STURT STREET ADELAIDE SA 5000
 6-14 BLUEVIEW PLACE LOVELY BANKS EAST VIC 3213   6-14 BLUEVIEW PLACE LOVELY BANKS VIC 3213
SHOP 30 490 OXLEY AVE PEREGIAN SPRINGS QLD 4020     490-492 OXLEY AVENUE REDCLIFFE QLD 4020
       835 MANDURAH ROAD NORTH BALDIVIS WA 6171      831-851 MANDURAH ROAD BALDIVIS WA 6171
        25/50 MANCHESTER ST MERRYLANDS NSW 2704 48-50 MANCHESTER STREET MERRYLANDS NSW 2160
    UNIT 9 1-3 ENDEAVOU ROAD CARINGBAH NSW 2229       1-3 ENDEAVOUR ROAD CARINGBAH NSW 2229
       485 ST KILDAS RD MELBOURNE VICTORIA 3004    485-489 ST KILDA ROAD MELBOURNE VIC 3004
    23-29 SALTASH STREET YAGOONA SOUTH NSW 2199       23-29 SALTASH STREET YAGOONA NSW 2199



fmt_interior_number rows: 8,090
klass
true_interior    1.0
unparsed         0.0

sample TRUE interior positives:
                                         dirty                                           anchor
      835 MANDURAH ROAD NORTH BALDIVIS WA 6171           831-851 MANDURAH ROAD BALDIVIS WA 6171
       25/50 MANCHESTER ST MERRYLANDS NSW 2704      48-50 MANCHESTER STREET MERRYLANDS NSW 2160
     14/471 GILBERT ROAD PRESTON EAST VIC 3082            469-471 GILBERT ROAD PRESTON VIC 3072
4 PICNIC POINT ESPLANADE MAROOCHYDORE QLD 4758 2-4 PICNIC POINT ESPLANADE MAROOCHYDORE QLD 4558
            142 BELMORE ROAD EVELEIGH NSW 2031           130-148 BELMORE ROAD RANDWICK NSW 2031
          36 GEORGE STREET LEICHNARDT NSW 2040          30-40 GEORGE STREET LEICHHARDT NSW 2040


In [ ]:
# Cell 11: contamination and coverage check. Run after 3_train_retriever.ipynb,
# which writes output_stage1/stage1_recall_per_query.tsv. Prints only.
# A. Contamination: test answers that also appear as training anchors.
# B. Coverage: which number of a span the fmt_interior_number rows write, the
#    last ("50" for 46-50) or a middle one ("32" for 22-36).

import pandas as pd, numpy as np, re
from pathlib import Path

SYN_PQ = Path("output_synthesis/synthetic_addresses.parquet")
PERQ   = Path("output_stage1/stage1_recall_per_query.tsv")
assert SYN_PQ.exists(), f"{SYN_PQ} not found; run cell 9 first"

syn = pd.read_parquet(SYN_PQ).fillna("")
syn_pos = syn[syn.label == "POS"]
print(f"{len(syn_pos):,} synthetic positives loaded\n")

# A. Contamination
if not PERQ.exists():
    print("A. CONTAMINATION: skipped, no stage1_recall_per_query.tsv found\n")
else:
    perq = pd.read_csv(PERQ, sep="\t", dtype={"address": str, "truth_pid": str})

    # render each test answer exactly as clean_string in cell 8 renders an
    # anchor, so that a real collision is found
    con.register("test_pid_df", pd.DataFrame({"pid": perq.truth_pid.unique()}))
    truth_rendered = con.sql(r"""
        SELECT k.pid,
               regexp_replace(trim(concat_ws(' ',
                   CASE WHEN g.number_last IS NOT NULL AND g.number_last <> ''
                             AND regexp_matches(k.k, '^[0-9]')
                        THEN k.k || '-' || g.number_last ELSE k.k END,
                   k.street_n, k.type_n, k.suffix_n,
                   k.locality_n, k.state_n, k.postcode)),
                   '\s+', ' ', 'g') AS rendered
        FROM test_pid_df t
        JOIN gnaf_key k ON k.pid = t.pid
        LEFT JOIN gnaf_range g ON g.pid = k.pid AND g.number_first = k.k
        QUALIFY row_number() OVER (PARTITION BY k.pid
            ORDER BY CASE WHEN k.k LIKE 'LOT %' THEN 1 ELSE 0 END, k.k) = 1
    """).df()
    R = dict(zip(truth_rendered.pid, truth_rendered.rendered.str.upper().str.strip()))

    syn_anchors = set(syn_pos.anchor.str.upper().str.strip())
    perq["truth_rendered"] = perq.truth_pid.map(R)
    hit = perq.truth_rendered.isin(syn_anchors) & perq.truth_rendered.notna()

    print("A. CONTAMINATION: test answers that also appear as training anchors")
    print(f"   {int(hit.sum())} of {len(perq)} test queries ({hit.mean()*100:.2f}%)")
    if hit.sum():
        print("\n   the affected rows (check whether they are range records):")
        for r in perq[hit].head(15).itertuples():
            print(f"     {r.address[:48]:<50} -> {r.truth_rendered[:44]}")
        if hit.sum() > 15:
            print(f"     ... and {int(hit.sum()) - 15} more")
    print("   Under ~1% is chance overlap. Several percent,\n"
          "   concentrated on range records, means the oversampling is leaking\n"
          "   the test set and the range recall figure cannot be trusted.\n")

# B. Coverage, with the same house-number rule as cell 10b
def house_number(dirty):
    s = str(dirty).upper().strip()
    s = re.sub(r"^(?:UNIT|LEVEL|SHOP|SUITE|LOT)\s+\d+[A-Z]?\s+", "", s)
    s = re.sub(r"^\d+[A-Z]?\s*/\s*", "", s)
    m = re.match(r"(\d+)", s)
    return int(m.group(1)) if m else None

inter = syn_pos[syn_pos.ops.str.contains("fmt_interior_number", regex=False)].copy()

def shape(row):
    n = house_number(row.dirty)
    rm = re.match(r"^(?:LOT\s+)?(\d+)-(\d+)", str(row.anchor).upper().strip())
    if n is None or not rm: return "unparsed"
    lo, hi = int(rm.group(1)), int(rm.group(2))
    if n == lo:      return "wrote_first"
    if n == hi:      return "wrote_last"       # 50 for 46-50
    if lo < n < hi:  return "wrote_middle"     # 32 for 22-36
    return "outside"

inter["shape"] = inter.apply(shape, axis=1)
counts = inter["shape"].value_counts()
print("B. COVERAGE SHAPE of the 'wrote one number of a span' training rows")
print(f"   {len(inter):,} POS rows carry fmt_interior_number\n")
for k in ["wrote_last", "wrote_middle", "wrote_first", "outside", "unparsed"]:
    n = int(counts.get(k, 0))
    print(f"     {k:14s} {n:>6,}  ({n/max(len(inter),1)*100:5.1f}%)")

print("\n   The test set contains 53 'wrote_last' and 20 'wrote_middle'\n"
      "   queries, so both patterns need real representation here. A few\n"
      "   hundred of the smaller one is enough to learn from; near zero is not.")

if int(counts.get("wrote_middle", 0)) < 200:
    print("\n   WARNING: wrote_middle is thin. Most G-NAF spans are two numbers\n"
          "   wide (46-50), where the only candidate after the parity filter IS\n"
          "   the last number, so endpoints dominate naturally. If this count is\n"
          "   near zero the 20 true-interior test queries stay uncovered.")

# span widths in these rows, which drive the split above
w = inter.anchor.str.extract(r"^(?:LOT\s+)?(\d+)-(\d+)").astype(float)
width = (w[1] - w[0]).dropna()
print(f"\n   span widths in these rows: median {width.median():.0f}, "
      f"{(width <= 2).mean()*100:.0f}% are 2 wide or less")

84,971 synthetic positives loaded



A. CONTAMINATION: test answers that also appear as training anchors
   0 of 785 test queries (0.00%)
   Under ~1% is chance overlap. Several percent,
   concentrated on range records, means the oversampling is leaking
   the test set and the range recall figure cannot be trusted.



B. COVERAGE SHAPE of the 'wrote one number of a span' training rows
   8,090 POS rows carry fmt_interior_number

     wrote_last      4,985  ( 61.6%)
     wrote_middle    3,102  ( 38.3%)
     wrote_first         0  (  0.0%)
     outside             0  (  0.0%)
     unparsed            3  (  0.0%)

   The test set contains 53 'wrote_last' and 20 'wrote_middle'
   queries, so both patterns need real representation here. A few
   hundred of the smaller one is enough to learn from; near zero is not.

   span widths in these rows: median 4, 44% are 2 wide or less


## Known limitations

- `fmt_single_vs_range` and the range sampler in cell 8b need the `gnaf_range` table. Setting `COLMAP["range_table"] = ""` turns both off.
- Some operations can only fire on a record that qualifies (a SAINT to abbreviate, a range to shorten). Their rates are fitted given eligibility, and cell 10 reports the eligible counts and how often each error operator was refused. The MOUNT and PORT rate is fitted on 3 eligible test-set rows and the SAINT rate on none.
- `P_INTERIOR`, `P_RANGE_NOHYPHEN`, `P_RANGE_SLASH`, `NEG_FRAC` and the 55 to 45 split of the two minor negative recipes are set by hand, not fitted.
- `err_loc_sublocal` uses an alias table when `COLMAP["alias_table"]` is set, and two patterns otherwise: drop a gazetted tail word such as BEACH or GROVE, or append a colloquial one such as VILLAGE or POINT.
- The lot number added by `fmt_prefix_lot` is made up.
- The character tables (`char_edit_ops.parquet`, `substitutions.tsv`) were written by the harvest before the manual label review, so they do not reflect the corrected labels.
- The records are a uniform sample of G-NAF, apart from the range oversampling in cell 8b. G-NAF is not distributed like the addresses in legal documents. Cell 10's lot-number counts show whether the generator adds or drops lot numbers.